# Stat 153/248 - Lab 9

In this notebook we'll work through a third dataset on Heart Rate Variability to add to what we discussed in class (see Lecture18.ipynb notebook).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal
import warnings
warnings.filterwarnings('ignore')

# statsmodels imports
import statsmodels.api as sm
from statsmodels.tsa.stattools import acf, pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning
warnings.filterwarnings('ignore', category=ConvergenceWarning)

plt.rcParams.update({
    'figure.figsize': (12, 4),
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 16,
})

## Heart Rate Variability

Heart rate variability (HRV) — the beat-to-beat fluctuation in your heart's rhythm — is a widely studied biosignal. The RR interval series (time between consecutive heartbeats) reflects a mix of:

- **Autonomic feedback loops** (sympathetic/parasympathetic regulation, which could be modeled like an AR component)
- **Respiratory and other short-lived perturbations** (breathing, posture changes, which could cause MA-like shocks)

This mix makes HRV a natural ARMA process. Let's see it in action.

We'll use data from the **MIT-BIH Normal Sinus Rhythm Database** on PhysioNet. We can get this from the package [WFDB (Waveform Database)](https://www.physionet.org/content/wfdb/10.7.0/)


In [ ]:
# Heart Rate Variability - RR intervals
!pip install wfdb

import wfdb
# MIT-BIH Normal Sinus Rhythm Database, record 16265
record = wfdb.rdrecord('16265', pn_dir='nsrdb/', channels=[0], 
                       sampfrom=0, sampto=128*600)  # ~10 min at 128 Hz
ecg = record.p_signal[:, 0]

# First we will plot the original ECG waveform. We will use this to measure
# the heart rate variability by finding peaks in the signal (the R wave - which
# is the first upward deflection in the ECG time series.)
timepts = np.linspace(0,600,128*600)
plt.plot(timepts[1:1000], ecg[1:1000])
plt.xlabel('Time (s)')

Next we will detect R-peaks in this ECG waveform and find their intervals.

In [ ]:
# Detect R-peaks
from scipy.signal import find_peaks

peaks, _ = find_peaks(ecg, distance=int(0.5*128), height=0.5)
rr_intervals = np.diff(peaks) / 128.0 * 1000  # in ms
hrv_data = rr_intervals
data_source = "PhysioNet MIT-BIH Normal Sinus Rhythm DB (record 16265)"

print(f"Data source: {data_source}")
print(f"Number of RR intervals: {len(hrv_data)}")
print(f"Mean RR: {np.mean(hrv_data):.1f} ms ({60000/np.mean(hrv_data):.0f} bpm)")
print(f"SDNN: {np.std(hrv_data):.1f} ms")

fig, axes = plt.subplots(2, 1, figsize=(14, 6))

axes[0].plot(hrv_data, linewidth=0.6)
axes[0].set(xlabel='Beat number', ylabel='RR interval (ms)', 
            title='RR Interval Time Series (Heart Rate Variability)')

# Also show a zoomed window
window = slice(700, 900)
axes[1].plot(np.arange(700, 900), hrv_data[window], 'o-', markersize=3, linewidth=0.8)
axes[1].set(xlabel='Beat number', ylabel='RR interval (ms)',
            title='Zoomed: beats 700-900')

plt.tight_layout()
plt.show()


### Predict (discussion)

Let's think about the structure of this time series. Discuss with a partner:

1. **Look at the zoomed plot.** Do you see features that look like slow drifts (AR) *and* sudden jumps (MA)?
2. **Predict the ACF shape.** Will it look like sunspots (oscillatory decay)? Gas prices (monotone decay)? Something else?
3. **Predict the PACF.** Will it cut off cleanly?


In [ ]:
# Demean for stationarity
y_hrv = hrv_data - np.mean(hrv_data)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

plot_acf(y_hrv, lags=40, ax=axes[0], title='ACF — HRV (RR intervals)')
plot_pacf(y_hrv, lags=40, ax=axes[1], title='PACF — HRV (RR intervals)', method='ywm')

plt.tight_layout()
plt.show()


### More predictions

Now having seen the ACF and the PACF, what is your best guess for the model? AR(p)? MA(q)? ARMA(p,q)? What orders?


## Results

The ACF decays but not as cleanly as pure AR would predict. The PACF doesn't cut off cleanly either, so this doesn't seem to be a pure AR *or* MA process.

This is what you might expect from ARMA. When you see gradual ACF decay *and* a messy PACF that doesn't cut off cleanly after a certain number of lags, the data likely has both autoregressive feedback and moving-average shock structure.

Let's try each of the models as before.

## AR vs MA vs ARMA

Here we will compare several AR, MA, and ARMA models of various orders. We will use the very naive split that we showed in class, where we will just try to predict the last `t` samples.

In [ ]:
t = 10 # Let's try and predict the last t samples

# Split our data into training and test data
y_train = y_hrv[:-t]
y_test = y_hrv[-t:]

print(f"Train: {len(y_train)} obs, Test: {len(y_test)} obs")

models_to_test = {
    'AR(1)':     (1, 0, 0),
    'AR(2)':     (2, 0, 0),
    'AR(3)':     (3, 0, 0),
    'AR(4)':     (4, 0, 0),
    'MA(1)':     (0, 0, 1),
    'MA(2)':     (0, 0, 2),
    'MA(4)':     (0, 0, 4),
    'ARMA(1,1)': (1, 0, 1),
    'ARMA(2,1)': (2, 0, 1),
    'ARMA(3,1)': (3, 0, 1),
}

results = {}
for name, order in models_to_test.items():
    model = ARIMA(y_train, order=order).fit()
    forecasts = model.forecast(steps=len(y_test))
    errors = y_test - forecasts
    results[name] = {
        'rmse': np.sqrt(np.mean(errors**2)),
        'mae': np.mean(np.abs(errors)),
        'forecasts': forecasts,
        'errors': errors,
    }
    print(f"{name:<12} RMSE={results[name]['rmse']:.5f}")



## Question:

* What do you notice about these models? 
* Which is the best in terms of RMSE?
* Does this depend on your train/test split? Try different values of t and see what you find

In [ ]:
fig, axes = plt.subplots(len(results), 1, figsize=(14, 2 * len(results)), sharex=True)

split_point = len(y_train)

for ax, (name, res) in zip(axes, results.items()):
    # Training data
    ax.plot(np.arange(len(y_train)), y_train, 'k-', linewidth=0.4, alpha=0.5, label='Train')
    # Test actual
    ax.plot(np.arange(split_point, split_point + len(y_test)), y_test, 'k-', linewidth=0.8, label='Actual')
    # Forecast
    ax.plot(np.arange(split_point, split_point + len(y_test)), res['forecasts'], 'r--', linewidth=1.2, label=f'{name} forecast')
    # Split line
    ax.axvline(split_point, color='orange', linewidth=1.5, linestyle=':', label='Train/Test split')
    ax.set(ylabel='RR interval', title=f'{name}  |  RMSE={res["rmse"]:.5f}')
    ax.legend(loc='upper left')
    ax.set_xlim(600,1000)

axes[-1].set(xlabel='Time index')
plt.tight_layout()
plt.show()

## Model parameter counts

Now let's look at model performance as a function of number of parameters. What do you notice about the comparison between AR, MA, and ARMA models for the same number of parameters for this dataset?

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

colors = {'AR': '#2196F3', 'MA': '#FF9800', 'ARMA': '#4CAF50'}

for name, res in results.items():
    # Determine model type and parameter count
    order = models_to_test[name]
    p, d, q = order
    nparams = p + q
    mtype = 'ARMA' if (p > 0 and q > 0) else ('AR' if p > 0 else 'MA')
    
    ax.scatter(nparams, res['rmse'], c=colors[mtype], s=100, zorder=3, edgecolors='white')
    ax.annotate(name, (nparams, res['rmse']), textcoords="offset points", xytext=(6, 6), fontsize=10)

# Legend (one entry per type)
for mtype, color in colors.items():
    if any((mtype == 'AR' and 'AR(' in n and 'ARMA' not in n) or 
           (mtype == 'ARMA' and 'ARMA' in n) or
           (mtype == 'MA' and 'MA(' in n and 'ARMA' not in n)
           for n in results):
        ax.scatter([], [], c=color, s=100, label=mtype, edgecolors='white')

ax.set(xlabel='Number of parameters (p + q)', ylabel='RMSE (lower is better)',
       title='Model Comparison: Forecast Error vs. Parsimony')
ax.legend(fontsize=12)
ax.set_xticks([1,2,3,4])
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Discussion

Let's look at the plot above:

1. **Among pure AR models**, how many lags do you need to get competitive? Is that parsimonious?
2. **Among pure MA models**, can they match the AR performance? How many lags?
3. **Compare to ARMA** How many total parameters do the best ARMA models use compared to the best pure AR?
4. **The parsimony argument**: ARMA(2,1) uses 3 parameters. What's the best pure AR can do with 3 parameters? With 7? (you may need to run some more)


## Implementing rolling cross validation

As we discussed in class, our forecasts get worse the longer we try to project out over time. Another way we might want to look at this, therefore, is through rolling cross validation. Let's look at  this for the best model, ARMA(2,1).

In [ ]:
start_forecast = 700
order = (2,0,1)

results_rolling = {}
all_forecast = []
# Keep adding data to the forecast, starting our forecast at `start_forecast`
for idx, t in enumerate(np.arange(start_forecast,len(y_hrv))):
    y_train = y_hrv[:t]
    y_test = y_hrv[t]
    model = ARIMA(y_train, order=order).fit()
    forecasts = model.forecast(steps=1)
    all_forecast.append(forecasts)
    errors = y_test - forecasts
    results_rolling = {
        'rmse': np.sqrt(np.mean(errors**2)),
        'mae': np.mean(np.abs(errors)),
        'forecasts': forecasts,
        'errors': errors,
    }
    print(f"RMSE={results_rolling['rmse']:.5f}")
    


In [ ]:
# Plot the data and forecast 
plt.figure()
plt.plot(np.arange(len(y_hrv[:start_forecast]), len(y_hrv[:start_forecast])+len(y_hrv[start_forecast:])), y_hrv[start_forecast:], label='true data')
plt.plot(np.arange(len(y_hrv[:start_forecast]), len(y_hrv[:start_forecast])+len(all_forecast)), all_forecast, label='forecast')
plt.plot(y_hrv[:start_forecast])
plt.axvline(start_forecast, color='r', linestyle='--')
plt.legend()

# Plot the forecast only (to zoom in)
plt.figure()
plt.plot(y_hrv[start_forecast:], label='true data')
plt.plot(all_forecast, label='forecast')

## To try yourself

Try doing this again with some of the worse models -- MA only, AR only, or various other combinations. What do you notice? How do the forecasts break down? What happens if you use a fixed sliding window for the training set? Does that help?

### Why is ARMA good for modeling HRV?

Think about what's generating heart rate variability:

**AR component (autonomic feedback):**  
Your heart rate at beat $t$ depends on recent beats through something called the baroreceptor reflex and autonomic regulation. If your heart speeds up, the nervous system must push it back slowly over several beats. This is autoregressive feedback: $x_t$ depends on $x_{t-1}, x_{t-2}$.

**MA component (transient perturbations):**  
Each breath modulates heart rate (respiratory sinus arrhythmia). A single breath is a *shock* that affects the current beat and the next beat or two, then vanishes. The current value depends on the current and recent shocks $w_t, w_{t-1}$.

**ARMA captures both**: slow autonomic feedback (AR) + fast respiratory/transient shocks (MA). 

This decomposition into "feedback" vs "shock propagation" is broadly useful across multiple fields:
- **Economics**: monetary policy (slow AR feedback) + supply shocks (transient MA)
- **Neuroscience**: ongoing neural dynamics (AR) + stimulus-evoked responses (MA)  
- **Climate**: ocean thermal inertia (AR) + weather perturbations (MA)